# Case Study §5 — SFT with COMPLETION-ONLY loss

Runnable twin of [`05_sft.py`](05_sft.py). The key contrast with §3:
- **§3 CPT** → full causal loss, **~100% of tokens** contribute.
- **§5 SFT** → prompt→completion data, loss on the **completion only**; the prompt is **masked** (−100),
  so the unmasked fraction is well under 100%.

We print that fraction to *prove* the mask is working. Forget `completion_only_loss=True` and the model
learns to generate questions too — the classic "echoes the prompt back" bug (`PITFALLS.md`).

> `MODE="trial"` uses ~10 pairs + a few steps (answers may be empty — barely trained). `"full"` for real.

In [ ]:
MODE = "trial"      # "trial" or "full"
INIT = "instruct"  # "instruct" or "base"
FORCE = False
import importlib.util, pathlib, sys, json
HERE = pathlib.Path.cwd()
root = next(p for p in [HERE, HERE/'scripts', HERE.parent/'scripts'] if (p/'config.py').exists())
sys.path.insert(0, str(root))
import config; config.set_mode(MODE)
spec = importlib.util.spec_from_file_location("s5", root / "05_sft.py")
s5 = importlib.util.module_from_spec(spec); spec.loader.exec_module(s5)
print(f"mode={config.RUN_MODE} init={INIT}")

## The data shape
Each example is `{prompt, completion}` — TRL computes loss on the completion only by default for this shape.

In [ ]:
rows = s5.load_seed(3)
for r in rows:
    print('prompt    :', r['prompt'])
    print('completion:', r['completion'][:80], '...')
    print()

## Run SFT
Self-sufficient (builds the seed Q&A if missing) and idempotent. Prints the **unmasked-token fraction**
(the proof that the prompt is masked) and a few sample answers at the end.

In [ ]:
m = s5.run_sft(init=INIT, force=FORCE)
print(json.dumps({k: m[k] for k in ['init','n_examples','unmasked_fraction','completion_only_loss']}, indent=2))

## Verify
The mask must be partial (prompt masked) — clearly different from CPT's ~100%.

In [ ]:
assert m['completion_only_loss'] is True
assert 0.0 < m['unmasked_fraction'] < 1.0, 'completion-only loss must mask the prompt (fraction < 100%)'
print(f"\u2713 §5 verified: unmasked {m['unmasked_fraction']*100:.0f}% (prompt masked) vs ~100% for CPT in \u00a73.")
print('Next: \u00a76 base-vs-instruct sweep over SFT-set size \u2014 the centerpiece experiment.')